> 在线讲义：[第 03 章 · 从存储程序到数据流执行：存储层次就像大楼里的厕所](https://wikieden.github.io/ai-infra-from-constraints/lessons/03-dataflow-hierarchy.html) · 本书《从约束推导 AI Infra》

# 第 3 章 · 从存储程序到数据流执行：存储层次就像大楼里的厕所

配套讲义：仓库里 `ai-infra/notes/lessons/03-dataflow-hierarchy.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。纯 Python，不需要 GPU，也不下载模型。练习题不在这里算，先在讲义里自己做。

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 H100 存储层次 | 容量对比 | `LEVELS` | 每 SM 256 KB … 显存 80 GB |
| 第二步 $n = 4$ 不分块 | $2n^3$ | `far_reads(4)` | 128 |
| 第二步 $n = 4$、$t = 2$ / $t = 4$ | $2n^3 / t$ | `far_reads(4, 2)`、`far_reads(4, 4)` | 64、32 |
| 第二步 逐块模拟核对公式 | 一块块数 | `simulate_tiled(4, 2)` | 64 |
| 第三步 MoE 平均 | $64 \times 2 / 8$ | `moe_avg(64, 8, 2)` | 16 |

H100 SXM 的容量取自公开规格（与《CUDA 从线程到 Tensor Core》第 5 章一致）；n = 4 的矩阵和 MoE 的 64 个 token、8 个专家是编的。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
LEVELS = [
    ("寄存器", 256 * 1024, "每 SM"),
    ("shared memory + L1", 256 * 1024, "每 SM"),
    ("L2 缓存", 50 * 10**6, "全卡"),
    ("显存 HBM", 80 * 10**9, "全卡"),
]

**函数 `far_reads`**：n×n 矩阵乘法从远处（显存）读多少次：不分块 2n³，分块 2n³/t。

In [ ]:
def far_reads(n: int, t: int = 1) -> int:
    """n×n 矩阵乘法从远处（显存）读多少次：不分块 2n³，分块 2n³/t。"""
    return 2 * n**3 // t

**函数 `simulate_tiled`**：一块一块数：每个 C 块沿 K 方向滑 n/t 次，每次搬 A、B 各一个 t×t 块。

In [ ]:
def simulate_tiled(n: int, t: int) -> int:
    """一块一块数：每个 C 块沿 K 方向滑 n/t 次，每次搬 A、B 各一个 t×t 块。"""
    reads = 0
    for _ in range(n // t):  # C 的块行
        for _ in range(n // t):  # C 的块列
            for _ in range(n // t):  # 沿 K 方向滑动
                reads += 2 * t * t  # A、B 各搬一个 t×t 块进近处
    return reads

**函数 `moe_avg`**：MoE 平均每个专家分到几个 token：tokens × k ÷ 专家数。

In [ ]:
def moe_avg(tokens: int, experts: int, k: int) -> float:
    """MoE 平均每个专家分到几个 token：tokens × k ÷ 专家数。"""
    return tokens * k / experts

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步：H100 SXM 的存储层次（越往下越大、越远）")
for name, size, scope in LEVELS:
    print(f"  {name:20s} {scope} {size:>15,} 字节")

In [ ]:
print("\n第二步：n = 4 的矩阵乘法，从显存读多少次")
print(f"  不分块 2n³ = {far_reads(4)}")
for t in (2, 4):
    print(f"  分块 t = {t}：2n³/t = {far_reads(4, t)}，逐块数一遍 = {simulate_tiled(4, t)}")

In [ ]:
print("\n第三步：MoE 平均每个专家分到几个 token（编的：64 个 token、8 个专家、每个选 2 个）")
print(f"  64 × 2 / 8 = {moe_avg(64, 8, 2):.0f}")

## 3. 代码里才看得到的

1. `simulate_tiled` 是三重循环一块块数出来的，和公式 $2n^3/t$ 对上，说明公式没有漏算；要求 $n$ 能被 $t$ 整除，否则边缘块要单独处理。
2. $t = n$ 时远处读取最少，但近处要同时放下两个 $n \times n$ 块；真实 kernel 的 $t$ 受 shared memory 容量限制（每 SM 最多 228 KB）。
3. MoE 的 `moe_avg` 只给平均数；真实路由的分布要看具体 token，代码里没有模拟不均匀。

## 4. 自己改一改

- 把 `far_reads(8, t)` 的 $t$ 依次取 1、2、4、8，看远处读取怎么减半。
- 算一算 $t = 32$ 时，两个 float32 的 $t \times t$ 块要多少 KB，和每 SM 的 shared memory 比一比。
- 把 `moe_avg` 的专家数从 8 改成 64，看平均每个专家还剩几个 token。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 厕所修在楼层边，近的小来远的宽；
> 读一次来用多遍，分块缓冲写成链；
> prefill、decode、KV、专家，复用各不同，一套缓存管不全。